# Turnwise Qwen3.5 development training

Train the pinned `unsloth/Qwen3.5-0.8B` decision model on 36 reviewed training snapshots. Select checkpoints using the eight validation snapshots, then calibrate on the five reserved calibration snapshots.

The 27 final-test snapshots from ten held-out conversations are not loaded during training, validation, checkpoint selection, or calibration. The final-test cell is manually gated and requires the frozen run configuration plus a separately prepared test export. Evaluate once; do not make further model or configuration choices from test results.

This notebook uses Turnwise's structured `state` / `questions` / `gold` decision format, not a generative text-prompt approximation. Upload only the development training, validation, and calibration exports until the final configuration is frozen. The task is synthetic and small; metrics are exploratory, not population-level evidence.

## 1. Colab runtime setup and dependencies

Use a Tesla T4 GPU runtime. These pins match the observed smoke-test stack and required evaluation/data libraries. Unsloth currently constrains the `datasets` version; this notebook reads the pinned local JSONL files directly and does not invoke dataset folder builders or download remote training data. The cell checks the PyTorch/CUDA/GPU combination before loading the model; stop and revalidate rather than silently changing the hardware recipe.

In [ ]:
%pip install -q \
    --extra-index-url https://download.pytorch.org/whl/cu128 \
    "torch==2.8.0+cu128" "torchvision==0.23.0+cu128" "torchaudio==2.8.0+cu128"
%pip install -q \
    "triton==3.4.0" \
    "unsloth==2026.11.7" \
    "unsloth_zoo==2026.11.7" \
    "transformers==5.2.0" \
    "trl==0.22.2" \
    "xformers==0.0.32.post2" \
    "datasets==4.6.1" \
    "peft==0.18.0" \
    "accelerate==1.15.0" \
    "evaluate==0.4.5" \
    "bitsandbytes==0.49.1" \
    "pandas==2.2.3" \
    "scikit-learn==1.8.0"

import accelerate
import bitsandbytes
import datasets
import evaluate
import json
import peft
import torch
import transformers
import triton
import trl
import unsloth
import unsloth_zoo
import xformers

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
assert torch.__version__.startswith("2.8.0"), f"This recipe was smoke-tested with PyTorch 2.8.0; found {torch.__version__}"
assert torch.version.cuda == "12.8", f"This recipe was smoke-tested with CUDA 12.8; found {torch.version.cuda}"
assert torch.cuda.get_device_capability(0) == (7, 5), "Select a Tesla T4 or revalidate this experiment on the new GPU."
SOFTWARE_VERSIONS = {
    "python": __import__("platform").python_version(),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "triton": triton.__version__, "unsloth": unsloth.__version__,
    "unsloth_zoo": unsloth_zoo.__version__, "transformers": transformers.__version__,
    "trl": trl.__version__, "xformers": xformers.__version__,
    "datasets": datasets.__version__, "peft": peft.__version__,
    "accelerate": accelerate.__version__, "evaluate": evaluate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
}
print("GPU:", torch.cuda.get_device_name(0))
print("Free / total VRAM:", tuple(round(value / 2**30, 2) for value in torch.cuda.mem_get_info()), "GiB")
print("Software versions:", json.dumps(SOFTWARE_VERSIONS, indent=2))

## 2. Project configuration and reproducibility

This experiment follows the already-reviewed 30/5/5/10 conversation split (36/8/5 development snapshots and 27 final-test snapshots). Do not randomize snapshots: prefixes and related scenarios stay grouped by their assigned conversation family. Fixed seeds improve repeatability; GPU kernels may still be nondeterministic.

In [ ]:
import json
import random
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import torch

MODEL_ID = "unsloth/Qwen3.5-0.8B"
MODEL_REVISION = "23c69c53358a07516b5827588b3fdb12ae78fd65"
SEED = 3407
CONFIG = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "max_seq_length": 2048,
    "num_train_epochs": 2,
    "train_batch_size": 1,
    "eval_batch_size": 1,
    "gradient_accumulation_steps": 1,
    "learning_rate": 2e-4,
    "head_learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "selection_metric": "minimum validation loss at epoch boundary",
    "seed": SEED,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DRIVE_ROOT = Path("/content/drive/MyDrive/TurnwiseDevelopment")
RUN_ID = datetime.now(timezone.utc).strftime("qwen35-08b-%Y%m%dT%H%M%SZ") + f"-{uuid4().hex[:8]}"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
print("Run directory:", RUN_DIR)
print("Configuration:", json.dumps(CONFIG, indent=2))

## 3. Upload the reviewed development exports

In the upload dialog select exactly the training and validation JSONL files plus their metadata sidecars. Keep calibration separate until the validation-selected model is fixed. Do not upload or mount final-test files in the training phase.

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
RUN_DIR.mkdir(parents=True, exist_ok=False)
INPUT_DIR = RUN_DIR / "development-inputs"
INPUT_DIR.mkdir()

required_development_files = {
    "development-training.jsonl",
    "development-training.metadata.json",
    "development-validation.jsonl",
    "development-validation.metadata.json",
}
uploaded_development = files.upload()
assert set(uploaded_development) == required_development_files, "Upload exactly the training and validation JSONL exports and sidecars."
assert not any("test" in name.lower() or "calibration" in name.lower() for name in uploaded_development), "Keep calibration and final-test data out of this upload."
for name in required_development_files:
    (INPUT_DIR / name).write_bytes(uploaded_development[name])
print("Saved only training and validation exports under", INPUT_DIR)

## 4. Validate the fixed split and build its index

The partition is already assigned and locked in the repository: 36 training snapshots from 30 conversations, eight validation snapshots from five, and five calibration snapshots from five. This notebook does not reshuffle or repartition. It verifies both JSONL and sidecar hashes, schemas, target-option membership, unique snapshots, and no family overlap.

In [ ]:
import hashlib
import json
import pandas as pd

EXPECTED_EXPORTS = {
    "training": {
        "rows": 36, "conversations": 30,
        "jsonl": "047cd69b878e454bbba97ca6e4b93c1d58022622865c93cd607aeb9fc306b72b",
        "metadata": "307f945e00934d2bfd54224e5d7ff7a071bcc9c2389579a060798f94844e27ad",
    },
    "validation": {
        "rows": 8, "conversations": 5,
        "jsonl": "d0216f37f996afed8ad73d823c71fcf1be498e5d8e7aaa71e4d8e543b5edab58",
        "metadata": "c177637ccb7fe8368a8b4eab9982bb59f5d5753f93c46664d53d606c5910cea9",
    },
    "calibration": {
        "rows": 5, "conversations": 5,
        "jsonl": "3275b94dc260edbea8f6ff934a06a1a1b1923416c2f7bc7276220e9c319159a8",
        "metadata": "6ead37b5afe80161e7f5cc1b556990d38c9b7b037112e62609fcccc83d5db897",
    },
}
DEVELOPMENT_MANIFEST_SHA256 = "c4dffdd57556a06a62b692682f5c84c5a33ba0717142afcb9c36e16b14a0310a"


def load_development_split(split, expected_rows, expected_conversations, expected_jsonl_sha256, expected_metadata_sha256):
    rows_path = INPUT_DIR / f"development-{split}.jsonl"
    metadata_path = INPUT_DIR / f"development-{split}.metadata.json"
    assert hashlib.sha256(rows_path.read_bytes()).hexdigest() == expected_jsonl_sha256, f"Unexpected {split} JSONL hash"
    assert hashlib.sha256(metadata_path.read_bytes()).hexdigest() == expected_metadata_sha256, f"Unexpected {split} sidecar hash"
    sidecar = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert sidecar["manifest_sha256"] == DEVELOPMENT_MANIFEST_SHA256
    assert sidecar["split_filter"] == split
    assert sidecar["row_count"] == expected_rows == len(sidecar["rows"])
    rows = [json.loads(line) for line in rows_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(rows) == expected_rows
    case_ids = set()
    snapshot_keys = set()
    for index, (row, provenance) in enumerate(zip(rows, sidecar["rows"])):
        assert set(row) == {"state", "questions", "gold"}
        assert "gold" not in row["state"]
        assert provenance["row_index"] == index and provenance["split"] == split
        assert row["state"]["conversation_id"] == provenance["case_id"]
        assert len(row["questions"]) == len(row["gold"]) == 6
        assert row["questions"].keys() == row["gold"].keys()
        for question_id, target in row["gold"].items():
            assert target in row["questions"][question_id]["criteria"]
        case_ids.add(provenance["case_id"])
        snapshot_keys.add((provenance["case_id"], provenance["snapshot_id"]))
    assert len(case_ids) == expected_conversations
    assert len(snapshot_keys) == expected_rows
    return rows, sidecar["rows"]

train_spec = EXPECTED_EXPORTS["training"]
validation_spec = EXPECTED_EXPORTS["validation"]
train_rows, train_metadata = load_development_split(
    "training", train_spec["rows"], train_spec["conversations"], train_spec["jsonl"], train_spec["metadata"]
)
validation_rows, validation_metadata = load_development_split(
    "validation", validation_spec["rows"], validation_spec["conversations"], validation_spec["jsonl"], validation_spec["metadata"]
)

index_rows = [
    {"split": split, "row_index": item["row_index"], "conversation_id": item["case_id"],
     "snapshot_id": item["snapshot_id"], "through_event_id": item["through_event_id"],
     "scenario_family_id": item["scenario_family_id"], "field_count": len(row["gold"]),
     "gold_fields": sorted(row["gold"])}
    for split, rows, metadata in (("training", train_rows, train_metadata), ("validation", validation_rows, validation_metadata))
    for row, item in zip(rows, metadata)
]
snapshot_index = pd.DataFrame(index_rows)
assert snapshot_index.groupby("split").size().to_dict() == {"training": 36, "validation": 8}
assert snapshot_index.groupby("split")["conversation_id"].nunique().to_dict() == {"training": 30, "validation": 5}
train_families = set(snapshot_index.loc[snapshot_index.split == "training", "scenario_family_id"])
validation_families = set(snapshot_index.loc[snapshot_index.split == "validation", "scenario_family_id"])
assert not (train_families & validation_families), "Scenario family crosses training and validation"
snapshot_index.to_json(RUN_DIR / "development-split-index.json", orient="records", indent=2)
split_manifest = {
    "development_manifest_sha256": DEVELOPMENT_MANIFEST_SHA256,
    "fixed_partition": True,
    "random_repartitioned": False,
    "final_test_loaded": False,
    "training": train_metadata,
    "validation": validation_metadata,
}
(RUN_DIR / "split-manifest.json").write_text(json.dumps(split_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(snapshot_index.groupby("split").agg(snapshots=("snapshot_id", "count"), conversations=("conversation_id", "nunique")))

In [ ]:
assert set(item["case_id"] for item in train_metadata).isdisjoint(
    item["case_id"] for item in validation_metadata
)
assert {item["split"] for item in train_metadata} == {"training"}
assert {item["split"] for item in validation_metadata} == {"validation"}
print("Training and validation conversation assignments are disjoint.")

## 5. Build the decision dataset

Turnwise exports each annotated transcript prefix as `state`, `questions`, and `gold`; Unsloth's decision encoder builds the option-scoring inputs from these records. There is no separate free-text prompt target. The gold choices are used only by the training/evaluation loss, never passed to `predict`.

## 6. Load Qwen3.5 0.8B and configure efficient fine-tuning

Load the same immutable revision used in the smoke run. Attach rank-16 LoRA and the decision head after the dataset is verified. Record resolved package versions and T4 memory; do not enable fp16/bf16 for the observed Qwen3.5 T4 configuration.

In [ ]:
import transformers
import unsloth
import torch
from unsloth import FastDecisionModel

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
assert torch.__version__.startswith("2.8.0"), f"Expected the smoke-tested PyTorch 2.8.0 runtime, found {torch.__version__}"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__, "Unsloth:", unsloth.__version__)

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL_ID,
    revision=MODEL_REVISION,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
)
resolved_revision = getattr(model.encoder.config, "_commit_hash", None)
print("Resolved encoder commit:", resolved_revision)
assert resolved_revision == MODEL_REVISION, "Loaded model commit is not the pinned revision"
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free GPU memory after load: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")

In [ ]:
train_items, train_build_report = FastDecisionModel.build_dataset(train_rows, tokenizer, model)
validation_items, validation_build_report = FastDecisionModel.build_dataset(validation_rows, tokenizer, model)

assert len(train_items) == 36
assert train_build_report.get("total") == 216
assert train_build_report.get("skipped") == train_build_report.get("truncated") == 0
assert len(validation_items) == 8
assert validation_build_report.get("total") == 48
assert validation_build_report.get("skipped") == validation_build_report.get("truncated") == 0

initial_validation_metrics = FastDecisionModel.evaluate(model, tokenizer, validation_items)
print("Prepared training snapshots/decisions:", len(train_items), train_build_report["total"])
print("Prepared validation snapshots/decisions:", len(validation_items), validation_build_report["total"])
print("Fresh-head validation baseline (diagnostic only):", initial_validation_metrics)


### Add LoRA adapters

The decision head and LoRA adapters train; Qwen3.5's T4 path uses float32 activations because fp16/bf16 are unsuitable or unavailable in the verified smoke runtime.

In [ ]:
import torch
from unsloth import FastDecisionModel

lora_names = [name for name, _ in model.named_parameters() if "lora_" in name]
assert not lora_names, "LoRA is already attached; reload the pinned base model before starting a fresh run."
model = FastDecisionModel.get_peft_model(
    model,
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print("Trainable parameters:", trainable_parameters)
print("Trainable LoRA tensors:", sum("lora_" in name for name, parameter in model.named_parameters() if parameter.requires_grad))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free GPU memory: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")

## 7. Train with periodic validation

Train the 36 preassigned training snapshots for two epochs. Evaluate on all eight validation snapshots after every epoch and let Trainer retain the lowest validation-loss checkpoint. The final test is not part of this Trainer or its inputs.

In [ ]:
from unsloth import DecisionTrainer
from transformers import TrainingArguments

TRAINING_CONFIG = CONFIG.copy()

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "trainer-checkpoints"),
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    per_device_train_batch_size=TRAINING_CONFIG["train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["eval_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=1,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=False,
    bf16=False,
)

trainer = DecisionTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_items,
    eval_dataset=validation_items,
    args=training_args,
    head_learning_rate=TRAINING_CONFIG["head_learning_rate"],
)
train_result = trainer.train()
print("Training metrics:", train_result.metrics)
print("Global steps:", trainer.state.global_step)
assert trainer.state.global_step > 0
assert trainer.state.best_model_checkpoint is not None

In [ ]:
FastDecisionModel.for_inference(model)
validation_metrics = FastDecisionModel.evaluate(model, tokenizer, validation_items)
print("Selected checkpoint:", trainer.state.best_model_checkpoint)
print("Minimum validation loss:", trainer.state.best_metric)
print("Validation decision metrics:", validation_metrics)
assert trainer.state.best_model_checkpoint is not None, "No validation-selected checkpoint was recorded"
assert trainer.state.best_metric is not None, "No validation loss was recorded"


## 8. Select and export the best checkpoint

`load_best_model_at_end` selects the epoch checkpoint with lowest validation loss. Save the selected adapter and decision head plus the aligned split manifest and run configuration to Drive. Do not select or compare checkpoints using calibration or final-test results.

In [ ]:
SELECTED_DIR = RUN_DIR / "selected-validation-checkpoint"
model.save_pretrained(str(SELECTED_DIR))
tokenizer.save_pretrained(str(SELECTED_DIR))

RUN_CONFIG = {
    "experiment_id": "repair-intake-qwen35-08b-development",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "task_version": "0.1.0",
    "development_manifest_sha256": "c4dffdd57556a06a62b692682f5c84c5a33ba0717142afcb9c36e16b14a0310a",
    "training_jsonl_sha256": "047cd69b878e454bbba97ca6e4b93c1d58022622865c93cd607aeb9fc306b72b",
    "validation_jsonl_sha256": "d0216f37f996afed8ad73d823c71fcf1be498e5d8e7aaa71e4d8e543b5edab58",
    "selected_by": "minimum validation loss at epoch boundary",
    "validation_metrics": validation_metrics,
    "training_arguments": TRAINING_CONFIG,
    "software_versions": SOFTWARE_VERSIONS,
    "runtime": {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda},
    "selected_checkpoint": str(SELECTED_DIR),
    "final_test_accessed": False,
}
print("Selected validation checkpoint:", SELECTED_DIR)
print("Best validation loss:", trainer.state.best_metric)

## 9. Calibration on the reserved split

This section runs only after the best validation-loss checkpoint is selected. It asks for the five calibration snapshots separately, estimates Unsloth's post-hoc calibration parameters, and saves the calibrated model. The calibration set is not used to train or choose the checkpoint; with only five snapshots, calibration estimates are exploratory.

In [ ]:
from google.colab import files as colab_files

uploaded_calibration = colab_files.upload()
required_calibration_files = {"development-calibration.jsonl", "development-calibration.metadata.json"}
assert set(uploaded_calibration) == required_calibration_files, "Upload exactly the reserved calibration export and sidecar."
for name in required_calibration_files:
    (INPUT_DIR / name).write_bytes(uploaded_calibration[name])

calibration_spec = EXPECTED_EXPORTS["calibration"]
calibration_rows, calibration_metadata = load_development_split(
    "calibration", calibration_spec["rows"], calibration_spec["conversations"],
    calibration_spec["jsonl"], calibration_spec["metadata"],
)
assert set(item["case_id"] for item in calibration_metadata).isdisjoint(
    item["case_id"] for item in train_metadata + validation_metadata
)
calibration_items, calibration_report = FastDecisionModel.build_dataset(calibration_rows, tokenizer, model)
assert len(calibration_items) == 5
assert calibration_report.get("total") == 30
assert calibration_report.get("skipped") == calibration_report.get("truncated") == 0

calibration_metrics = FastDecisionModel.calibrate(model, tokenizer, calibration_items)
print("Calibration metrics:", calibration_metrics)

CALIBRATED_DIR = RUN_DIR / "calibrated-decision-model"
model.save_pretrained(str(CALIBRATED_DIR))
tokenizer.save_pretrained(str(CALIBRATED_DIR))
(RUN_DIR / "calibration-metrics.json").write_text(
    json.dumps(calibration_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

RUN_CONFIG.update({
    "calibration": {
        "split": "calibration",
        "rows": len(calibration_rows),
        "metrics": calibration_metrics,
        "procedure": "FastDecisionModel.calibrate after validation checkpoint selection",
    },
    "selected_checkpoint": str(CALIBRATED_DIR),
})
run_config_text = json.dumps(RUN_CONFIG, indent=2, sort_keys=True) + "\n"
RUN_CONFIG_SHA256 = hashlib.sha256(run_config_text.encode("utf-8")).hexdigest()
(RUN_DIR / "frozen-run-config.json").write_text(run_config_text, encoding="utf-8")
print("Frozen run-config SHA-256:", RUN_CONFIG_SHA256)
print("Calibrated checkpoint:", CALIBRATED_DIR)

In [ ]:
assert set(item["case_id"] for item in calibration_metadata).isdisjoint(
    item["case_id"] for item in train_metadata + validation_metadata
)
assert {item["scenario_family_id"] for item in calibration_metadata}.isdisjoint(
    {item["scenario_family_id"] for item in train_metadata + validation_metadata}
)
print("Calibration conversations and families are held out from training and validation.")

## 10. Final evaluation on held-out test

Do not run this section until the selected checkpoint, calibration procedure, metrics, and run configuration are frozen. Only then, from the repository root, create the isolated test export:

```bash
uv run turnwise export-unsloth \
  --manifest data/manifests/final-test.json \
  --task configs/tasks/repair-intake.yaml \
  --split test \
  --output data/test/exports/final-test.jsonl \
  --metadata data/test/exports/final-test.metadata.json
```

Upload only those two files when the final cell opens its upload dialog. The cell checks the locked manifest hash and split/counts, then performs one inference pass without supplying gold to `predict`. It saves metrics and errors in Drive; never tune, recalibrate, or rerun after seeing results. Do not commit per-example final-test results.

In [ ]:
RUN_FINAL_TEST = False
CONFIRM_FROZEN_CONFIG_SHA256 = ""

if not RUN_FINAL_TEST:
    print("Final test skipped. Leave this off until the run configuration is frozen and ready for its one-time evaluation.")
else:
    assert CONFIRM_FROZEN_CONFIG_SHA256 == RUN_CONFIG_SHA256, "Confirm the frozen run-config SHA-256 first."
    final_report_path = RUN_DIR / "final-test-results.json"
    final_test_marker = RUN_DIR / ".final-test-started"
    assert not final_test_marker.exists() and not final_report_path.exists(), "Final test was already started for this run; do not rerun it."

    from google.colab import files as colab_files

    uploaded_test = colab_files.upload()
    required_test_files = {"final-test.jsonl", "final-test.metadata.json"}
    assert set(uploaded_test) == required_test_files, "Upload only the separately prepared, locked final-test export and sidecar."
    test_input_dir = RUN_DIR / "final-test-input"
    test_input_dir.mkdir(parents=True, exist_ok=False)
    for name in required_test_files:
        (test_input_dir / name).write_bytes(uploaded_test[name])

    test_jsonl = test_input_dir / "final-test.jsonl"
    test_metadata_path = test_input_dir / "final-test.metadata.json"
    test_metadata = json.loads(test_metadata_path.read_text(encoding="utf-8"))
    assert test_metadata["manifest_sha256"] == "cb96053ba8356188cc6344815c6190eea30cb00e3b9f1e2fc832c71d6b0bd1eb"
    assert test_metadata["split_filter"] == "test"
    assert test_metadata["row_count"] == len(test_metadata["rows"]) == 27

    test_rows = [json.loads(line) for line in test_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(test_rows) == 27
    assert len(test_rows) == len(test_metadata["rows"])
    assert {item["split"] for item in test_metadata["rows"]} == {"test"}
    test_case_ids = {item["case_id"] for item in test_metadata["rows"]}
    assert len(test_case_ids) == 10
    development_case_ids = {item["case_id"] for item in train_metadata + validation_metadata + calibration_metadata}
    development_families = {item["scenario_family_id"] for item in train_metadata + validation_metadata + calibration_metadata}
    assert test_case_ids.isdisjoint(development_case_ids), "Test conversation overlaps development data"
    assert {item["scenario_family_id"] for item in test_metadata["rows"]}.isdisjoint(development_families), "Test family overlaps development data"
    assert all(row["state"]["conversation_id"] == item["case_id"] for row, item in zip(test_rows, test_metadata["rows"]))
    final_test_marker.write_text(RUN_CONFIG_SHA256 + "\n", encoding="utf-8")

    FastDecisionModel.for_inference(model)
    field_counts = {}
    field_confusions = {}
    errors = []
    for row, item in zip(test_rows, test_metadata["rows"]):
        answers = FastDecisionModel.predict(model, tokenizer, row["state"], row["questions"])
        assert set(answers) == set(row["gold"]) == set(row["questions"])
        for question_id, result in answers.items():
            expected = row["gold"][question_id]
            predicted = result["answer"]
            options = row["questions"][question_id]["criteria"]
            assert expected in options and predicted in options, question_id
            field = question_id.removeprefix("value__")
            entry = field_counts.setdefault(field, {"correct": 0, "total": 0, "gold_unknown": 0, "predicted_unknown": 0})
            entry["total"] += 1
            entry["correct"] += int(predicted == expected)
            entry["gold_unknown"] += int(expected == "__unknown__")
            entry["predicted_unknown"] += int(predicted == "__unknown__")
            field_confusions.setdefault(field, {}).setdefault(expected, {}).setdefault(predicted, 0)
            field_confusions[field][expected][predicted] += 1
            if predicted != expected:
                errors.append({"case_id": item["case_id"], "snapshot_id": item["snapshot_id"], "field": field, "gold": expected, "prediction": predicted})

    total_correct = sum(counts["correct"] for counts in field_counts.values())
    total_decisions = sum(counts["total"] for counts in field_counts.values())
    field_metrics = {
        field: {
            **counts,
            "accuracy": counts["correct"] / counts["total"],
            "prediction_coverage": 1 - counts["predicted_unknown"] / counts["total"],
        }
        for field, counts in field_counts.items()
    }
    final_results = {
        "run_config_sha256": RUN_CONFIG_SHA256,
        "test_manifest_sha256": test_metadata["manifest_sha256"],
        "test_jsonl_sha256": hashlib.sha256(test_jsonl.read_bytes()).hexdigest(),
        "snapshot_count": len(test_rows),
        "conversation_count": len(test_case_ids),
        "decision_count": total_decisions,
        "overall_decision_accuracy": total_correct / total_decisions,
        "field_metrics": field_metrics,
        "field_confusion_counts": field_confusions,
        "errors": errors,
        "interpretation_note": "Prefixes are correlated within conversations; report both snapshot and conversation counts.",
    }
    final_report_path.write_text(json.dumps(final_results, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    pd.DataFrame(errors, columns=["case_id", "snapshot_id", "field", "gold", "prediction"]).to_csv(RUN_DIR / "final-test-errors.csv", index=False)
    print(json.dumps({"snapshot_count": len(test_rows), "conversation_count": len(test_case_ids), "overall_decision_accuracy": final_results["overall_decision_accuracy"], "field_metrics": field_metrics, "error_count": len(errors), "saved_report": str(final_report_path)}, indent=2))
    print("Final test evaluated once. Do not use these results to change this run or select another checkpoint.")

## 11. Colab upload and artifact download

Training and validation exports are uploaded near the start; calibration is uploaded only after checkpoint selection; the final-test export is uploaded only after the explicit freeze gate. Model files and reports stay in the Drive run folder. Download a model/config bundle here without bundling raw labels or per-example final-test errors.

In [ ]:
import json
import zipfile
from google.colab import files as colab_files

bundle_path = RUN_DIR / f"{RUN_ID}-model-and-config.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(CALIBRATED_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(RUN_DIR))
    for name in ("frozen-run-config.json", "calibration-metrics.json", "split-manifest.json"):
        path = RUN_DIR / name
        if path.exists():
            bundle.write(path, path.relative_to(RUN_DIR))
    final_report_path = RUN_DIR / "final-test-results.json"
    if final_report_path.exists():
        final_results = json.loads(final_report_path.read_text(encoding="utf-8"))
        final_summary = {key: value for key, value in final_results.items() if key != "errors"}
        summary_path = RUN_DIR / "final-test-summary.json"
        summary_path.write_text(json.dumps(final_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        bundle.write(summary_path, summary_path.relative_to(RUN_DIR))
print("Saved bundle:", bundle_path)
colab_files.download(str(bundle_path))